<a href="https://colab.research.google.com/github/mobarakol/tutorial_notebooks/blob/main/Brain_Tumors_DenseNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Brain Tumor MRI Dataset
SRC: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset

In [ ]:
import gdown

url = 'https://drive.google.com/uc?id=1Oms9X0Vpid_kN8jiSgz-3MhRA5BcmivE'
gdown.download(url,'braintumor.zip',quiet=True)
!unzip -q braintumor.zip -d braintumor

Dataloader

In [ ]:
import os
import sys
import argparse
from torchvision import datasets
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
import torchvision.transforms as transforms


def get_arguments():
    parser = argparse.ArgumentParser(description='CIFAR-10H Training')
    parser.add_argument('--lr', default=1e-3, type=float, help='learning rate')
    parser.add_argument('--lr_schedule', default=0, type=int, help='lr scheduler')
    parser.add_argument('--batch_size', default=100, type=int, help='batch size')
    parser.add_argument('--test_batch_size', default=128, type=int, help='batch size')
    parser.add_argument('--num_epoch', default=20, type=int, help='epoch number')
    parser.add_argument('--num_classes', type=int, default=4, help='number classes')
    if 'ipykernel' in sys.modules:
        args = parser.parse_args([])
    else:
        args = parser.parse_args()
    return args

def train(model, trainloader, criterion, optimizer):
    model.train()
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def test(model, testloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return correct / total

if __name__ == '__main__':
    args = get_arguments()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225])

    train_dataset = datasets.ImageFolder(root='/content/braintumor/Training', transform=transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        normalize,
    ]))

    test_dataset = datasets.ImageFolder(root='/content/braintumor/Testing', transform=transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        normalize,
    ]))

    trainloader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=2)
    testloader = DataLoader(test_dataset, batch_size=args.test_batch_size, shuffle=False, num_workers=2)
    print('Training on:', device, 'train sample size:', len(train_dataset), 'test sample size:', len(test_dataset))
    print(train_dataset.class_to_idx)

    model = models.densenet121(pretrained=True)
    model.classifier = nn.Linear(model.classifier.in_features, args.num_classes)
    model.to(device)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr, betas=(0.9, 0.999), weight_decay=0.1)
    criterion = nn.CrossEntropyLoss()

    best_epoch, best_acc = 0.0, 0
    for epoch in range(args.num_epoch):
        train(model, trainloader, criterion, optimizer)
        accuracy = test(model, testloader)
        if accuracy > best_acc:
            patience = 0
            best_acc = accuracy
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model_densenet.pth.tar')
        print('epoch: {}  acc: {:.4f}  best epoch: {}  best acc: {:.4f}'.format(
                epoch, accuracy, best_epoch, best_acc, optimizer.param_groups[0]['lr']))


Training on: cuda train sample size: 5712 test sample size: 1311
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


epoch: 0  acc: 0.8368  best epoch: 0  best acc: 0.8368
epoch: 1  acc: 0.9626  best epoch: 1  best acc: 0.9626
epoch: 2  acc: 0.8436  best epoch: 1  best acc: 0.9626
epoch: 3  acc: 0.8658  best epoch: 1  best acc: 0.9626
epoch: 4  acc: 0.8284  best epoch: 1  best acc: 0.9626
epoch: 5  acc: 0.9382  best epoch: 1  best acc: 0.9626
epoch: 6  acc: 0.9001  best epoch: 1  best acc: 0.9626
epoch: 7  acc: 0.9847  best epoch: 7  best acc: 0.9847
epoch: 8  acc: 0.6857  best epoch: 7  best acc: 0.9847
epoch: 9  acc: 0.9718  best epoch: 7  best acc: 0.9847
epoch: 10  acc: 0.9779  best epoch: 7  best acc: 0.9847
epoch: 11  acc: 0.9596  best epoch: 7  best acc: 0.9847
epoch: 12  acc: 0.9870  best epoch: 12  best acc: 0.9870
epoch: 13  acc: 0.9756  best epoch: 12  best acc: 0.9870
epoch: 14  acc: 0.9886  best epoch: 14  best acc: 0.9886
epoch: 15  acc: 0.9893  best epoch: 15  best acc: 0.9893
epoch: 16  acc: 0.9939  best epoch: 16  best acc: 0.9939
epoch: 17  acc: 0.9954  best epoch: 17  best acc: 0.9